# 04 · Challenge the trained player
Run → Run All Cells. This notebook evaluates your saved model without changing its weights.

Random play is a weak baseline. Two deliberately different rule-based players challenge face damage and board control. They are simple benchmarks, not proven strong players.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "lab.py").exists(): ROOT = ROOT.parent
if not (ROOT / "lab.py").exists(): raise RuntimeError("Open this notebook inside death-knight-lab.")
sys.path.insert(0, str(ROOT))
from benchmarks.suite import load_model, run_suite, show_results
model_path, model_data, _, _ = load_model()
print("Evaluating:", model_path)
print("Training games:", model_data["trained_games"])

## Settings
200 games per opponent = 800 games total. Increase to 1,000 per opponent for narrower bounds. Interrupted runs resume saved games when settings stay the same. To explicitly pick an older model, set MODEL_PATH to its full path.

In [ ]:
GAMES_PER_OPPONENT = 200
SEED = 20260918
MODEL_PATH = model_path

## Balanced matches
Each independent block uses two randomly generated decks and four games: both assignments of decks, each going first and second. All opponents face the same schedule. The model only sees legal, visible information. No learning occurs here.

These new deck samples are independent of training, but exact deck duplicates remain possible. Repeatedly tuning to this suite makes it a development benchmark; reserve a fresh seed and opponents for a final evaluation.

In [ ]:
from IPython.display import display, HTML
from html import escape
import time
progress_display = display(HTML("Starting evaluation…"), display_id=True)
last_update = [0.0]
def progress(done, total, opponent):
    now = time.monotonic()
    if now-last_update[0] < .25 and done != total: return
    last_update[0] = now
    progress_display.update(HTML(f"<strong>{escape(opponent)}</strong><p>{done:,} / {total:,} games</p><progress value='{done}' max='{total}' style='width:100%;height:24px'></progress>"))
results, results_path = run_suite(MODEL_PATH, GAMES_PER_OPPONENT, SEED, progress)
show_results(results)
print("Saved results:", results_path)

## What the results mean
- Beating random play shows basic learning. It does not establish strong strategy.
- Losing to a rule-based opponent identifies a useful training challenge. Rules opponents have their own blind spots.
- Winning across this suite supports broader competence within this small simulator; it does not prove optimal play.
- Bounds are conservative and account for correlated games and all four comparisons. Draws count as zero wins; the saved report also includes half-credit scores.

## Building toward the best deck
Next: train against a mixture of opponents and archived versions of the learner, then retest on reserved opponents. After the player is stronger, compare and mutate decks with a frozen player and balanced matchups. Promising decks will also need equal opportunities for deck-specific training.

Full Standard requires a much larger, verified card simulator and opposing classes. The present 33-card Death Knight/neutral environment can identify only the strongest combinations we discover in this restricted setting. Deck evolution is not implemented in this notebook.